s# Find the Trained Model

Diagnostic notebook to locate where the unfrozen model was actually saved.

In [0]:
# Cell 0 - Check possible model locations

import os

possible_paths = [
    "/dbfs/FileStore/allen_brain_data/models/unfrozen",
    "/tmp/checkpoints/unfrozen",
    "dbfs:/FileStore/allen_brain_data/models/unfrozen",
    "/dbfs/FileStore/allen_brain_data/models",
    "/tmp/checkpoints"
]

print("Checking possible model locations:\n")
for path in possible_paths:
    exists = os.path.exists(path)
    status = "✅ EXISTS" if exists else "❌ NOT FOUND"
    print(f"{status}  {path}")
    
    if exists and os.path.isdir(path):
        contents = os.listdir(path)
        print(f"         Contents: {len(contents)} items")
        if 'checkpoint-' in str(contents) or 'config.json' in contents:
            print(f"         📁 Model files detected!")
    print()

Checking possible model locations:

✅ EXISTS  /dbfs/FileStore/allen_brain_data/models/unfrozen
         Contents: 3 items
         📁 Model files detected!

❌ NOT FOUND  /tmp/checkpoints/unfrozen

❌ NOT FOUND  dbfs:/FileStore/allen_brain_data/models/unfrozen

✅ EXISTS  /dbfs/FileStore/allen_brain_data/models
         Contents: 4 items

❌ NOT FOUND  /tmp/checkpoints



In [0]:
# Cell 1 - Search for checkpoint directories

import os
import glob

print("Searching for checkpoint directories in /tmp/checkpoints/...\n")

if os.path.exists("/tmp/checkpoints/unfrozen"):
    checkpoints = glob.glob("/tmp/checkpoints/unfrozen/checkpoint-*")
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for cp in sorted(checkpoints):
        print(f"  - {cp}")
        files = os.listdir(cp)
        print(f"    Files: {', '.join(sorted(files)[:5])}{'...' if len(files) > 5 else ''}")
    
    # Check root of unfrozen dir
    root_files = os.listdir("/tmp/checkpoints/unfrozen")
    print(f"\nRoot of /tmp/checkpoints/unfrozen:")
    print(f"  {len(root_files)} items: {', '.join(sorted(root_files)[:10])}{'...' if len(root_files) > 10 else ''}")
    
    # Check if config.json exists in root (indicates final model saved here)
    if 'config.json' in root_files:
        print(f"\n✅ FOUND FINAL MODEL at /tmp/checkpoints/unfrozen")
        print(f"   This is the model saved by trainer.save_model()")
else:
    print("/tmp/checkpoints/unfrozen does not exist")

Searching for checkpoint directories in /tmp/checkpoints/...

/tmp/checkpoints/unfrozen does not exist


In [0]:
# Cell 2 - Check DBFS FileStore

import os

print("Checking /dbfs/FileStore/allen_brain_data/...\n")

if os.path.exists("/dbfs/FileStore/allen_brain_data"):
    print("✅ /dbfs/FileStore/allen_brain_data exists")
    
    if os.path.exists("/dbfs/FileStore/allen_brain_data/models"):
        print("✅ /dbfs/FileStore/allen_brain_data/models exists")
        contents = os.listdir("/dbfs/FileStore/allen_brain_data/models")
        print(f"   Contents: {contents}")
    else:
        print("❌ /dbfs/FileStore/allen_brain_data/models does NOT exist")
        print("   The directory was never created!")
else:
    print("❌ /dbfs/FileStore/allen_brain_data does NOT exist")

Checking /dbfs/FileStore/allen_brain_data/...

✅ /dbfs/FileStore/allen_brain_data exists
✅ /dbfs/FileStore/allen_brain_data/models exists
   Contents: ['coarse_6class', 'depth2', 'full', 'unfrozen']


In [0]:
# Cell 3 - Determine what happened

print("DIAGNOSIS:\n")

import os

# Check where model actually is
if os.path.exists("/tmp/checkpoints/unfrozen/config.json"):
    print("✅ The model WAS saved successfully during training.")
    print("   Location: /tmp/checkpoints/unfrozen")
    print("\n⚠️  PROBLEM: trainer.save_model() saved to OUTPUT_DIR (/tmp/checkpoints/unfrozen)")
    print("   NOT to FINAL_MODEL_DIR (/dbfs/FileStore/allen_brain_data/models/unfrozen)")
    print("\n   The notebook has a bug in Cell 7:")
    print("   - It calls: trainer.save_model(FINAL_MODEL_DIR)")
    print("   - But Trainer already saved the final model to OUTPUT_DIR during training")
    print("   - The explicit save_model() call might have failed silently")
    print("\n📍 YOUR MODEL IS AT: /tmp/checkpoints/unfrozen")
    print("\n⚠️  WARNING: /tmp is ephemeral! If the cluster terminates, you'll lose the model!")
    print("\nNEXT STEPS:")
    print("1. Copy the model from /tmp to DBFS (persistent storage)")
    print("2. Then log it to MLflow")
    
elif os.path.exists("/dbfs/FileStore/allen_brain_data/models/unfrozen/config.json"):
    print("✅ The model is at the expected DBFS location.")
    print("   Location: /dbfs/FileStore/allen_brain_data/models/unfrozen")
    print("\n   You can proceed with logging it to MLflow.")
    
else:
    # Check for any checkpoint
    import glob
    checkpoints = glob.glob("/tmp/checkpoints/unfrozen/checkpoint-*/config.json")
    if checkpoints:
        latest = max(checkpoints)
        print(f"⚠️  No final model found, but found checkpoints.")
        print(f"\n   Latest checkpoint: {os.path.dirname(latest)}")
        print("\n   The training completed but the final model save failed.")
        print("   You can use the latest checkpoint as your model.")
    else:
        print("❌ CRITICAL: No model found anywhere!")
        print("\n   Neither /tmp nor /dbfs has the model.")
        print("   Check if:")
        print("   1. The training actually completed successfully")
        print("   2. The cluster is still running (if not, /tmp was wiped)")
        print("   3. There was an error in the model saving step")

DIAGNOSIS:

✅ The model is at the expected DBFS location.
   Location: /dbfs/FileStore/allen_brain_data/models/unfrozen

   You can proceed with logging it to MLflow.


In [0]:
# Cell 4 - Copy model from /tmp to DBFS (if needed)

import os
import shutil

SOURCE = "/tmp/checkpoints/unfrozen"
DEST = "/dbfs/FileStore/allen_brain_data/models/unfrozen"

if os.path.exists(f"{SOURCE}/config.json"):
    print(f"Copying model from {SOURCE} to {DEST}...")
    
    # Create parent directory
    os.makedirs(os.path.dirname(DEST), exist_ok=True)
    
    # Copy entire directory
    if os.path.exists(DEST):
        print(f"⚠️  Destination already exists. Removing old version...")
        shutil.rmtree(DEST)
    
    shutil.copytree(SOURCE, DEST)
    print(f"✅ Model copied to {DEST}")
    
    # Verify
    files = os.listdir(DEST)
    print(f"\nVerification:")
    print(f"  Files in {DEST}: {len(files)}")
    print(f"  config.json exists: {os.path.exists(f'{DEST}/config.json')}")
    print(f"  model.safetensors exists: {os.path.exists(f'{DEST}/model.safetensors')}")
    
    print(f"\n✅ Ready to log to MLflow! Update the mlflow_log_unfrozen_model.ipynb notebook.")
else:
    print(f"❌ Source model not found at {SOURCE}")
    print(f"   Cannot copy. Run the diagnostic cells above to find where the model is.")

❌ Source model not found at /tmp/checkpoints/unfrozen
   Cannot copy. Run the diagnostic cells above to find where the model is.
